<a href="https://colab.research.google.com/github/Zidane86-06/Data_Engineering_workshop/blob/main/Day5/Day5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
print('All libraries imported successfully!')

All libraries imported successfully!


In [38]:
import pandas as pd
df=pd.read_csv('student_performance.csv')
print(f'Dataset loaded:{df.shape[0]} students,{df.shape[1]} columns')
print(f'Columns:{df.columns.tolist()}')
print(f'\nMissing values:{df.isnull().sum().sum()}')
print(f'\nFirst 5 rows:')
df.head()

Dataset loaded:30 students,13 columns
Columns:['student_id', 'name', 'age', 'gender', 'department', 'semester', 'math_score', 'science_score', 'english_score', 'programming_score', 'attendance_percentage', 'city', 'admission_year']

Missing values:0

First 5 rows:


,student_id,name,age,gender,department,semester,math_score,science_score,english_score,programming_score,attendance_percentage,city,admission_year
0,1001,Aarav Sharma,19,Male,Computer Science,2,85,78,72,91,92,Mumbai,2023
1,1002,Priya Patel,20,Female,Computer Science,2,76,82,88,79,87,Ahmedabad,2023
2,1003,Rohit Verma,19,Male,Electronics,2,65,74,61,55,78,Delhi,2023
3,1004,Sneha Reddy,20,Female,Mechanical,2,70,80,75,48,95,Hyderabad,2023
4,1005,Arjun Nair,19,Male,Computer Science,2,92,88,81,95,90,Kochi,2023


In [39]:
df_ml=df.copy()
le_gender=LabelEncoder()
df_ml['gender_encoded']=le_gender.fit_transform(df_ml['gender'])
print(f'Gender encoding:{dict(zip(le_gender.classes_,le_gender.transform(le_gender.classes_)))}')
le_depts=LabelEncoder()
df_ml['dept_encoded']=le_depts.fit_transform(df_ml['department'])
print(f'Department encoding:{dict(zip(le_depts.classes_,le_depts.transform(le_depts.classes_)))}')
print('\nNew columns added:gender_encoded,department_encoded')
df_ml[['gender','gender_encoded','department','dept_encoded']].head(5)

Gender encoding:{'Female': np.int64(0), 'Male': np.int64(1)}
Department encoding:{'Civil': np.int64(0), 'Computer Science': np.int64(1), 'Electronics': np.int64(2), 'Mechanical': np.int64(3)}

New columns added:gender_encoded,department_encoded


,gender,gender_encoded,department,dept_encoded
0,Male,1,Computer Science,1
1,Female,0,Computer Science,1
2,Male,1,Electronics,2
3,Female,0,Mechanical,3
4,Male,1,Computer Science,1


In [40]:
feature_cols=[
    'math_score',
    'science_score',
    'english_score',
    'attendance_percentage',
    'gender_encoded',
    'dept_encoded'
]
x=df_ml[feature_cols]
y=df_ml['programming_score']
print(f'Feature matrix :{x.shape} (students x features)')
print(f'Target vector y shape:{y.shape} (one score per student))')
print(f'\nFeature columns:{feature_cols}')
print(f'Target column:programming_score')
print(f'\nTarget range:{y.min()} to {y.max()} (mean: {y.mean():.1f})')

Feature matrix :(30, 6) (students x features)
Target vector y shape:(30,) (one score per student))

Feature columns:['math_score', 'science_score', 'english_score', 'attendance_percentage', 'gender_encoded', 'dept_encoded']
Target column:programming_score

Target range:38 to 97 (mean: 67.6)


In [41]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(
    x,y,
    test_size=0.2,
    random_state=42
)
print(f'Total students :{len(x)}')
print(f'Training students:{len(x_train)} ({len(x_train)/len(x)*100:.0f}%)')
print(f'Testing students:{len(x_test)} ({len(x_test)/len(x)*100:.0f}%)')
print(f'Training target mean:{y_train.mean():.1f}')
print(f'Testing target mean:{y_test.mean():.1f}')

Total students :30
Training students:24 (80%)
Testing students:6 (20%)
Training target mean:68.0
Testing target mean:65.8


In [42]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

lr_model=LinearRegression()
lr_model.fit(x_train_scaled,y_train)
lr_pred=lr_model.predict(x_test_scaled)
lr_mae=mean_absolute_error(y_test,lr_pred)
lr_mse=mean_squared_error(y_test,lr_pred)
lr_r2=r2_score(y_test,lr_pred)
print(f'Linear Regression MAE:{lr_mae:.2f}')
print(f'Linear Regression MSE:{lr_mse:.2f}')
print(f'Linear Regression R2:{lr_r2:.2f}')
print()
print('Learned coefficients:')
for feature,coef in zip(feature_cols,lr_model.coef_):
    print(f'{feature}:{coef:.2f}')
print(f'{"bias (intercept)":<28}: {lr_model.intercept_:+.3f}')

Linear Regression MAE:9.37
Linear Regression MSE:131.54
Linear Regression R2:0.74

Learned coefficients:
math_score:20.26
science_score:7.61
english_score:2.40
attendance_percentage:-12.89
gender_encoded:-0.40
dept_encoded:-0.45
bias (intercept)            : +68.042


In [43]:
import numpy as np

dt_model = DecisionTreeRegressor(max_depth=5, random_state=42)
dt_model.fit(x_train_scaled, y_train)
dt_pred = dt_model.predict(x_test_scaled)
dt_mae = mean_absolute_error(y_test, dt_pred)
dt_rmse = np.sqrt(mean_squared_error(y_test, dt_pred))
dt_r2 = r2_score(y_test, dt_pred)

print('=== Model 2: Decision Tree (max_depth=5) ===')
print(f'MAE : {dt_mae:.2f}')
print(f'RMSE : {dt_rmse:.2f}')
print(f'R2 : {dt_r2:.4f}')

=== Model 2: Decision Tree (max_depth=5) ===
MAE : 10.75
RMSE : 16.08
R2 : 0.4803
